In [17]:
import numpy as np
import pandas as pd
import joblib
import json
from sklearn.ensemble import RandomForestRegressor

In [ ]:
df = pd.read_csv("../dataset/v2.csv",parse_dates=["week"])
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
C:\Users\User\AppData\Local\Temp\ipykernel_26976\2105816834.py:1: SyntaxWarning: invalid escape sequence '\ '
  df = pd.read_csv("zyntra-flow\forecasting-service\ dataset\v2.csv",parse_dates=["week"])


OSError: [Errno 22] Invalid argument: 'zyntra-flow\x0corecasting-service\\ dataset\x0b2.csv'

In [ ]:
df = df.sort_values(["province","master_product_id","week"]).reset_index(drop=True)
df.head()

,snapshot_id,master_product_id,province,week,total_units_sold
0,0b103d12-a30c-4862-a0f1-9b67618c96eb,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-06,118
1,92a2cac1-4231-491e-8c84-733e87711252,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-13,131
2,fa7a2f9e-165e-4c7a-8668-a48d86a9dc35,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-20,154
3,708dd097-0c48-491f-a884-ee432531fbd1,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-27,154
4,2206ca62-e8ba-49ff-a923-6993fa777ac3,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-03,162


In [ ]:
group_cols = ["province","master_product_id"]

In [ ]:
for lag in [1,2,3,4]:
    df[f"lag_{lag}"] = df.groupby(group_cols)["total_units_sold"].shift(lag)

In [ ]:
df.head()

,snapshot_id,master_product_id,province,week,total_units_sold,lag_1,lag_2,lag_3,lag_4
0,0b103d12-a30c-4862-a0f1-9b67618c96eb,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-06,118,NaN,NaN,NaN,NaN
1,92a2cac1-4231-491e-8c84-733e87711252,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-13,131,118.0,NaN,NaN,NaN
2,fa7a2f9e-165e-4c7a-8668-a48d86a9dc35,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-20,154,131.0,118.0,NaN,NaN
3,708dd097-0c48-491f-a884-ee432531fbd1,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-27,154,154.0,131.0,118.0,NaN
4,2206ca62-e8ba-49ff-a923-6993fa777ac3,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-03,162,154.0,154.0,131.0,118.0


In [ ]:
df["rolling_mean_4"] = (
    df.groupby(group_cols)["total_units_sold"]
    .transform(lambda s: s.shift(1).rolling(4).mean())
)

In [ ]:
df.head()

,snapshot_id,master_product_id,province,week,total_units_sold,lag_1,lag_2,lag_3,lag_4,rolling_mean_4
0,0b103d12-a30c-4862-a0f1-9b67618c96eb,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-06,118,NaN,NaN,NaN,NaN,NaN
1,92a2cac1-4231-491e-8c84-733e87711252,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-13,131,118.0,NaN,NaN,NaN,NaN
2,fa7a2f9e-165e-4c7a-8668-a48d86a9dc35,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-20,154,131.0,118.0,NaN,NaN,NaN
3,708dd097-0c48-491f-a884-ee432531fbd1,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-27,154,154.0,131.0,118.0,NaN,NaN
4,2206ca62-e8ba-49ff-a923-6993fa777ac3,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-03,162,154.0,154.0,131.0,118.0,139.25


In [ ]:
df["rolling_std_4"] = (
    df.groupby(group_cols)["total_units_sold"]
    .transform(lambda s: s.shift(1).rolling(4).std())
)

In [ ]:
df.head()

,snapshot_id,master_product_id,province,week,total_units_sold,lag_1,lag_2,lag_3,lag_4,rolling_mean_4,rolling_std_4
0,0b103d12-a30c-4862-a0f1-9b67618c96eb,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-06,118,NaN,NaN,NaN,NaN,NaN,NaN
1,92a2cac1-4231-491e-8c84-733e87711252,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-13,131,118.0,NaN,NaN,NaN,NaN,NaN
2,fa7a2f9e-165e-4c7a-8668-a48d86a9dc35,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-20,154,131.0,118.0,NaN,NaN,NaN,NaN
3,708dd097-0c48-491f-a884-ee432531fbd1,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-27,154,154.0,131.0,118.0,NaN,NaN,NaN
4,2206ca62-e8ba-49ff-a923-6993fa777ac3,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-03,162,154.0,154.0,131.0,118.0,139.25,17.839563


In [ ]:
df["week_of_year"] = df["week"].dt.isocalendar().week.astype(int)

In [ ]:
df.head()

,snapshot_id,master_product_id,province,week,total_units_sold,lag_1,lag_2,lag_3,lag_4,rolling_mean_4,rolling_std_4,week_of_year
0,0b103d12-a30c-4862-a0f1-9b67618c96eb,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-06,118,NaN,NaN,NaN,NaN,NaN,NaN,2
1,92a2cac1-4231-491e-8c84-733e87711252,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-13,131,118.0,NaN,NaN,NaN,NaN,NaN,3
2,fa7a2f9e-165e-4c7a-8668-a48d86a9dc35,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-20,154,131.0,118.0,NaN,NaN,NaN,NaN,4
3,708dd097-0c48-491f-a884-ee432531fbd1,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-27,154,154.0,131.0,118.0,NaN,NaN,NaN,5
4,2206ca62-e8ba-49ff-a923-6993fa777ac3,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-03,162,154.0,154.0,131.0,118.0,139.25,17.839563,6


In [ ]:
df["month"] = df["week"].dt.month

In [ ]:
df.head()

,snapshot_id,master_product_id,province,week,total_units_sold,lag_1,lag_2,lag_3,lag_4,rolling_mean_4,rolling_std_4,week_of_year,month
0,0b103d12-a30c-4862-a0f1-9b67618c96eb,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-06,118,NaN,NaN,NaN,NaN,NaN,NaN,2,1
1,92a2cac1-4231-491e-8c84-733e87711252,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-13,131,118.0,NaN,NaN,NaN,NaN,NaN,3,1
2,fa7a2f9e-165e-4c7a-8668-a48d86a9dc35,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-20,154,131.0,118.0,NaN,NaN,NaN,NaN,4,1
3,708dd097-0c48-491f-a884-ee432531fbd1,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-27,154,154.0,131.0,118.0,NaN,NaN,NaN,5,1
4,2206ca62-e8ba-49ff-a923-6993fa777ac3,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-03,162,154.0,154.0,131.0,118.0,139.25,17.839563,6,2


In [ ]:
province_map = {p: i for i,p in enumerate(sorted(df["province"].unique()))}

In [ ]:
province_map

{'Central': 0,
 'Eastern': 1,
 'North Central': 2,
 'North Western': 3,
 'Northern': 4,
 'Sabaragamuwa': 5,
 'Southern': 6,
 'Uva': 7,
 'Western': 8}

In [ ]:
product_map = {p: i for i,p in enumerate(sorted(df["master_product_id"].unique()))}

In [ ]:
product_map

{'261ca6b2-4391-4072-bcd6-8165b18c54ac': 0,
 '26813f9a-be13-45a2-97b6-d50f3a112ec1': 1,
 '3d1b091f-8769-4577-897e-d6678e667241': 2,
 '405d509b-3bd5-471d-a034-3589abd6ae97': 3,
 '41f509e8-bfce-4eb6-ae96-c7f4a1fb9f58': 4,
 '5103423a-d764-4a5d-8808-129b757891df': 5,
 '59b94e1a-c6cc-47b1-ac9d-ba7c8b925901': 6,
 '5b4c6e64-3170-44f8-90dd-c9b579f49373': 7,
 '65d8d58a-3bd7-4c24-a59b-990581601c04': 8,
 '7c577546-74cb-4542-86ef-e93fc8ec3bfd': 9,
 '8c4aadc0-e757-45c4-b2a9-fe79b28ec8d4': 10,
 '9906ad8f-b4a2-479f-9d1d-375ac82ece80': 11,
 'a8830609-9603-4bd1-a696-da81ab905da5': 12,
 'a920457a-8f10-49e8-a142-ebb145b662c2': 13,
 'afdd0dad-7ab2-423e-a699-15ac765394a2': 14,
 'b06a6fca-e4b5-41b3-a1cd-2312c801da1b': 15,
 'c6904871-a603-424a-842d-49e7c030141e': 16,
 'd65ba2ea-765d-42fc-9e06-cfb15a045a2e': 17,
 'e5ca2484-deb0-415f-b409-41788a93ab96': 18,
 'e647ac48-e2e0-43bb-a5e5-65a73c2e89ea': 19,
 'ee1aa16a-087d-4e1a-8166-7e0dd3212eb2': 20,
 'fb2b3db7-810a-46e5-bd9d-1b6cff9e4b71': 21}

In [ ]:
df["province_enc"] = df["province"].map(province_map)
df["product_enc"] = df["master_product_id"].map(product_map)

In [ ]:
df.head()

,snapshot_id,master_product_id,province,week,total_units_sold,lag_1,lag_2,lag_3,lag_4,rolling_mean_4,rolling_std_4,week_of_year,month,province_enc,product_enc
0,0b103d12-a30c-4862-a0f1-9b67618c96eb,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-06,118,NaN,NaN,NaN,NaN,NaN,NaN,2,1,0,0
1,92a2cac1-4231-491e-8c84-733e87711252,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-13,131,118.0,NaN,NaN,NaN,NaN,NaN,3,1,0,0
2,fa7a2f9e-165e-4c7a-8668-a48d86a9dc35,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-20,154,131.0,118.0,NaN,NaN,NaN,NaN,4,1,0,0
3,708dd097-0c48-491f-a884-ee432531fbd1,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-01-27,154,154.0,131.0,118.0,NaN,NaN,NaN,5,1,0,0
4,2206ca62-e8ba-49ff-a923-6993fa777ac3,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-03,162,154.0,154.0,131.0,118.0,139.25,17.839563,6,2,0,0


In [ ]:
features_cols = [
    "lag_1","lag_2","lag_3","lag_4",
    "rolling_mean_4","rolling_std_4",
    "week_of_year","month",
    "province_enc","product_enc"
]

target_col = "total_units_sold"

In [ ]:
train_df = df.dropna(subset=features_cols)

In [ ]:
train_df.head()

,snapshot_id,master_product_id,province,week,total_units_sold,lag_1,lag_2,lag_3,lag_4,rolling_mean_4,rolling_std_4,week_of_year,month,province_enc,product_enc
4,2206ca62-e8ba-49ff-a923-6993fa777ac3,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-03,162,154.0,154.0,131.0,118.0,139.25,17.839563,6,2,0,0
5,db952cfb-66ee-459a-8b28-cd6002f64e99,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-10,164,162.0,154.0,154.0,131.0,150.25,13.375973,7,2,0,0
6,cab71a1a-c29f-44a6-896e-a15365203db6,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-17,155,164.0,162.0,154.0,154.0,158.50,5.259911,8,2,0,0
7,b3bd43a7-2743-431e-9cd2-688bfb68f2e7,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-02-24,144,155.0,164.0,162.0,154.0,158.75,4.991660,9,2,0,0
8,2cd0f545-f139-4bf0-9942-64fe8ad3191c,261ca6b2-4391-4072-bcd6-8165b18c54ac,Central,2025-03-03,124,144.0,155.0,164.0,162.0,156.25,9.032349,10,3,0,0


In [ ]:
X = train_df[features_cols]
y = train_df[target_col]

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=3,
    random_state=42
)
model.fit(X,y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [ ]:
joblib.dump(model,"../model/g-model.joblib")

['../model/g-model.joblib']

In [ ]:
with open("feature_columns.json","w") as f:
    json.dump({
        "feature_cols":features_cols,
        "province_map":province_map,
        "product_map":product_map,
    },f,indent=2)

In [ ]:
preds = model.predict(X)
mae = np.mean(np.abs(preds-y))
print(f"In-sample MAE: {mae:.2f}")

In-sample MAE: 21.28
